# テストデータを用いたカリキュラム追加学習（128model_Test と同じデータ）

このノートブックは、80kカリキュラム学習済みの **wavelet 128×256×256 モデル**を読み込み、`128model_Test.ipynb` と同じ `Data/TestData/<pair>/` の `.npz`（`Train` キー）を画像プールとして使い、元のカリキュラム学習と同じランダムDVF教師あり目的で追加学習します。

> 注意: 通常テストデータは最終評価専用です。このノートブックで学習に使ったデータに対する評価値は、汎化性能のテスト結果としては扱えません。評価用には別の未使用データを残してください。

実行順: 以下を上から順に実行し、最後のセルだけを実行してください。パス、エポック数、チェックポイント名は最後のセルの設定値で変更できます。


In [ ]:
import os
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Sequential
import torch.optim as optim
import voxelmorph as vxm
import neurite as ne
import scipy.ndimage

os.environ['VXM_BACKEND'] = 'pytorch'

In [ ]:
os.environ.get('VXM_BACKEND')

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
mse_loss = vxm.losses.MSE().loss
grad_loss = vxm.losses.Grad('l2').loss

def total_loss(y_true, y_pred):
    mse = mse_loss(y_true, y_pred)
    grad = grad_loss(y_true, y_pred)
    return mse + 0.01 * grad, mse, grad
#     return mse_loss(y_true, y_pred)

def MSE_Loss(y_true, y_pred):
    y_true = y_true.to(device)
    y_pred = y_pred.to(device)
    mse = mse_loss(y_true, y_pred)
    return mse

def lncc_loss(I, J, window=9, eps=1e-5):
    # I, J: (B, 1, D, H, W)
    padding = window // 2
    weight = torch.ones(1, 1, window, window, window, device=I.device)

    I2 = I * I
    J2 = J * J
    IJ = I * J

    I_sum = F.conv3d(I, weight, padding=padding)
    J_sum = F.conv3d(J, weight, padding=padding)
    I2_sum = F.conv3d(I2, weight, padding=padding)
    J2_sum = F.conv3d(J2, weight, padding=padding)
    IJ_sum = F.conv3d(IJ, weight, padding=padding)

    win_size = window ** 3
    u_I = I_sum / win_size
    u_J = J_sum / win_size

    cross = IJ_sum - u_J * I_sum - u_I * J_sum + u_I * u_J * win_size
    I_var = I2_sum - 2 * u_I * I_sum + u_I * u_I * win_size
    J_var = J2_sum - 2 * u_J * J_sum + u_J * u_J * win_size

    lncc = cross * cross / (I_var * J_var + eps)
    return -torch.mean(lncc)  # maximize LNCC → minimize -LNCC

In [ ]:
# configure unet input shape (concatenation of moving and fixed images)
ndim = 3
unet_input_features = 2
# inshape = (*x_train.shape[1:], unet_input_features)

nb_features = [
    [32, 64, 64, 64, 64],
    [64, 64, 64, 64, 64, 32, 16, 16]
]


In [ ]:
import voxelmorph as vxm
import inspect

print(vxm.__file__)
print(vxm.networks.__file__)
print([name for name in dir(vxm.networks) if "VxmDense" in name])

In [ ]:
model3D = vxm.networks.VxmDense_128_256_256((128, 256, 256), nb_features, int_steps=0)
model3D.to(device)
optimizer = optim.Adam(model3D.parameters(), lr=1e-4)

transformer = vxm.layers.SpatialTransformer((64, 128, 128)).to(device)
transformer256 = vxm.layers.SpatialTransformer((128, 256, 256)).to(device)

In [ ]:
import math
from pathlib import Path
import torch
import matplotlib.pyplot as plt

band_names = ['LLL', 'LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH']

wavelet_vis_enabled = False
wavelet_vis_every = 100
wavelet_vis_dir = Path('wavelet_stage_outputs')
wavelet_vis_dir.mkdir(exist_ok=True)

class Haar3DAnalysisOnly(nn.Module):
    def __init__(self):
        super().__init__()

        hL = torch.tensor([1.0, 1.0], dtype=torch.float32) / math.sqrt(2.0)
        hH = torch.tensor([1.0, -1.0], dtype=torch.float32) / math.sqrt(2.0)

        filters = []
        names = []

        for z_name, z_filter in zip(['L', 'H'], [hL, hH]):
            for y_name, y_filter in zip(['L', 'H'], [hL, hH]):
                for x_name, x_filter in zip(['L', 'H'], [hL, hH]):
                    kernel = (
                        z_filter[:, None, None]
                        * y_filter[None, :, None]
                        * x_filter[None, None, :]
                    )
                    filters.append(kernel)
                    names.append(z_name + y_name + x_name)

        weight = torch.stack(filters, dim=0).unsqueeze(1)
        self.register_buffer('weight', weight)
        self.names = names

    def forward(self, x):
        x = F.pad(x, (0, 1, 0, 1, 0, 1))
        return F.conv3d(x, self.weight, stride=1, padding=0)

def analysis_filter_3d(x, analysis_layer):
    return analysis_layer(x)

def down_sampling_3d(w):
    return w[:, :, ::2, ::2, ::2]

def up_sampling_3d(w_down):
    B, C, D, H, W = w_down.shape
    w_up = torch.zeros(
        B, C, D * 2, H * 2, W * 2,
        dtype=w_down.dtype,
        device=w_down.device
    )
    w_up[:, :, ::2, ::2, ::2] = w_down
    return w_up

def make_3d_filter(fz, fy, fx):
    return fz[:, None, None] * fy[None, :, None] * fx[None, None, :]

def create_synthesis_filters(device):
    low = torch.tensor([1.0, 1.0], dtype=torch.float32, device=device) / math.sqrt(2.0)
    high = torch.tensor([1.0, -1.0], dtype=torch.float32, device=device) / math.sqrt(2.0)

    filters = torch.stack([
        make_3d_filter(low, low, low),
        make_3d_filter(low, low, high),
        make_3d_filter(low, high, low),
        make_3d_filter(low, high, high),
        make_3d_filter(high, low, low),
        make_3d_filter(high, low, high),
        make_3d_filter(high, high, low),
        make_3d_filter(high, high, high),
    ], dim=0)

    filters = torch.flip(filters, dims=[1, 2, 3]).unsqueeze(1)
    return filters

def synthesis_filter_3d(w_up, synthesis_filters):
    B, C, D, H, W = w_up.shape
    filtered_bands = []

    for i in range(C):
        band = w_up[:, i:i + 1, :, :, :]
        kernel = synthesis_filters[i:i + 1]
        filtered = F.conv3d(band, kernel, stride=1, padding=1)
        filtered = filtered[:, :, :D, :H, :W]
        filtered_bands.append(filtered)

    filtered_bands = torch.cat(filtered_bands, dim=1)
    reconstructed = torch.sum(filtered_bands, dim=1, keepdim=True)
    return reconstructed, filtered_bands

analysis = Haar3DAnalysisOnly().to(device)
synthesis_filters = create_synthesis_filters(device)
analysis_names = analysis.names

In [ ]:
# TestData curriculum fine-tuning.
# Uses the same synthetic-DVF curriculum objective as the 80k pretraining:
#   100 * MSE(target_warped_image, reconstructed_warped_image)
# + 0.01 * MSE(teacher_lowres_DVF, predicted_lowres_DVF)
import csv
import time
from pathlib import Path
from tqdm.auto import tqdm

# ----- Settings -----
# Change only PROJECT_ROOT if you copied the notebook elsewhere.
PROJECT_ROOT = Path('/Users/michico/Documents/大和先輩修論/Saito')
DATA_ROOT = PROJECT_ROOT / 'Data/TestData'
# Set this to the final 80k curriculum checkpoint you actually produced.
PRETRAINED_MODEL_PATH = PROJECT_ROOT / (
    'curriculum_80k_stage_checkpoints/model_analysis_pipeline_pretrain_curriculum_final.pth'
)
OUTPUT_DIR = PROJECT_ROOT / 'testdata_curriculum_finetune_checkpoints'
TOTAL_EPOCHS = 20_000
STAGE_EPOCHS = 2_000       # stage 1 ... 10
BATCH_SIZE = 2
LEARNING_RATE = 1e-6
RANDOM_SEED = 20260816
SAVE_EVERY_STAGE = True

if TOTAL_EPOCHS % STAGE_EPOCHS != 0:
    raise ValueError('TOTAL_EPOCHS must be divisible by STAGE_EPOCHS.')
NUM_STAGES = TOTAL_EPOCHS // STAGE_EPOCHS
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Test data directory was not found: {DATA_ROOT.resolve()}')
if not PRETRAINED_MODEL_PATH.is_file():
    raise FileNotFoundError(
        'Curriculum checkpoint was not found. Set PRETRAINED_MODEL_PATH to your 80k weight file: '
        f'{PRETRAINED_MODEL_PATH.resolve()}'
    )


def to_n_dhw(array, label):
    """Convert one or more volumes to (N, 128, 256, 256)."""
    array = np.asarray(array, dtype=np.float32)
    if array.ndim == 3:
        array = array[np.newaxis, ...]
    if array.ndim != 4:
        raise ValueError(f'{label}: expected 3-D/4-D data, got {array.shape}')
    if array.shape[1:] == (128, 256, 256):
        return array
    if array.shape[:3] == (128, 256, 256):
        return np.transpose(array, (3, 0, 1, 2))
    if array.shape[1:] == (256, 256, 128):
        return np.transpose(array, (0, 3, 1, 2))
    if array.shape[:3] == (256, 256, 128):
        return np.transpose(array, (3, 2, 0, 1))
    raise ValueError(
        f'{label}: unsupported shape {array.shape}; expected a permutation of (N, 128, 256, 256). '
        'Do not use the 64x128x128 resized tensors here: this model requires original resolution.'
    )


def load_128model_testdata(data_root):
    """Read the exact TestData pair-folder structure used by 128model_Test.ipynb."""
    pair_dirs = sorted(path for path in data_root.iterdir() if path.is_dir())
    if not pair_dirs:
        raise FileNotFoundError(f'No pair folders were found under {data_root.resolve()}')

    volumes, used_files = [], []
    for pair_dir in pair_dirs:
        # 128model_Test.ipynb obtains its fixed/moving pair from the two npz files
        # in each folder. Both images are valid source images for synthetic DVF training.
        npz_files = sorted(pair_dir.glob('*.npz'))
        if len(npz_files) != 2:
            raise ValueError(
                f'{pair_dir}: expected exactly 2 .npz files like 128model_Test.ipynb, found {len(npz_files)}'
            )
        for npz_path in npz_files:
            with np.load(npz_path, allow_pickle=False) as archive:
                if 'Train' not in archive:
                    raise KeyError(f'{npz_path}: missing key "Train". Available keys: {archive.files}')
                volumes.append(to_n_dhw(archive['Train'], str(npz_path)))
                used_files.append(str(npz_path))

    result = np.concatenate(volumes, axis=0)
    if len(result) < BATCH_SIZE:
        raise ValueError(f'Only {len(result)} volumes were found; BATCH_SIZE={BATCH_SIZE} cannot be sampled.')
    print(f'Loaded {len(result)} source volumes from {len(pair_dirs)} TestData pairs.')
    print('Training volume shape:', result.shape)
    return result, used_files


def volume_batch_generator(volumes, batch_size, seed):
    rng = np.random.default_rng(seed)
    while True:
        indices = rng.integers(0, len(volumes), size=batch_size)
        yield torch.from_numpy(volumes[indices]).unsqueeze(1), indices


def gaussian_smooth_3d(tensor, sigma=2.0):
    # Kept equivalent to the original curriculum implementation.
    from scipy.ndimage import gaussian_filter
    smooth = gaussian_filter(tensor.detach().cpu().numpy(), sigma=[0, 0, sigma, sigma, sigma])
    return torch.tensor(smooth, dtype=torch.float32, device=tensor.device)


def curriculum_reconstruct(moving, target):
    moving_bands = down_sampling_3d(analysis_filter_3d(moving, analysis))
    target_bands = down_sampling_3d(analysis_filter_3d(target, analysis))
    predicted_flow = model3D(moving_bands, target_bands)
    warped_bands = torch.cat(
        [transformer(moving_bands[:, i:i + 1], predicted_flow) for i in range(moving_bands.shape[1])],
        dim=1,
    )
    reconstructed, _ = synthesis_filter_3d(up_sampling_3d(warped_bands), synthesis_filters)
    return reconstructed, predicted_flow


torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

test_volumes, source_files = load_128model_testdata(DATA_ROOT)
train_generator = volume_batch_generator(test_volumes, BATCH_SIZE, RANDOM_SEED)

# The class and shape must match the 80k wavelet curriculum checkpoint.
model3D = vxm.networks.VxmDense_128_256_256((128, 256, 256), nb_features, int_steps=0).to(device)
try:
    checkpoint = torch.load(PRETRAINED_MODEL_PATH, map_location=device, weights_only=True)
except TypeError:  # older PyTorch
    checkpoint = torch.load(PRETRAINED_MODEL_PATH, map_location=device)
state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
model3D.load_state_dict(state_dict)
optimizer = optim.Adam(model3D.parameters(), lr=LEARNING_RATE)
transformer = vxm.layers.SpatialTransformer((64, 128, 128)).to(device)
transformer256 = vxm.layers.SpatialTransformer((128, 256, 256)).to(device)
print(f'Loaded curriculum weights: {PRETRAINED_MODEL_PATH.resolve()}')

history = []
started_at = time.time()
for epoch in tqdm(range(1, TOTAL_EPOCHS + 1), desc='TestData curriculum fine-tuning'):
    # Same stage schedule as the original: maximum displacement increases by 1 px per stage.
    stage = (epoch - 1) // STAGE_EPOCHS + 1
    moving_cpu, source_indices = next(train_generator)
    moving = moving_cpu.to(device=device, dtype=torch.float32)

    lowres_teacher_flow = (torch.rand((BATCH_SIZE, 3, 8, 16, 16), device=device) * 2.0 - 1.0) * stage
    lowres_teacher_flow = gaussian_smooth_3d(lowres_teacher_flow, sigma=2.0)
    fullres_teacher_flow = F.interpolate(
        lowres_teacher_flow, size=(128, 256, 256), mode='trilinear', align_corners=False
    )
    teacher_flow_128 = F.interpolate(
        fullres_teacher_flow, size=(64, 128, 128), mode='trilinear', align_corners=False
    )
    target = transformer256(moving, fullres_teacher_flow)

    optimizer.zero_grad(set_to_none=True)
    reconstructed, predicted_flow = curriculum_reconstruct(moving, target)
    image_loss = MSE_Loss(target, reconstructed) * 100.0
    dvf_loss = MSE_Loss(teacher_flow_128, predicted_flow) * 0.01
    loss = image_loss + dvf_loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model3D.parameters(), max_norm=1.0)
    optimizer.step()

    row = {
        'epoch': epoch, 'stage': stage, 'max_shift_px': stage,
        'loss': loss.item(), 'image_loss': image_loss.item(), 'dvf_loss': dvf_loss.item(),
    }
    history.append(row)
    if epoch % 100 == 0:
        elapsed = time.time() - started_at
        eta = elapsed / epoch * (TOTAL_EPOCHS - epoch)
        print(
            f'Epoch {epoch:05d}/{TOTAL_EPOCHS}: stage={stage:02d}, max=±{stage}px, '
            f'loss={loss.item():.6f}, image={image_loss.item():.6f}, dvf={dvf_loss.item():.6f}, '
            f'source={source_indices.tolist()}, ETA={eta / 3600:.1f} h'
        )

    if SAVE_EVERY_STAGE and epoch % STAGE_EPOCHS == 0:
        checkpoint_path = OUTPUT_DIR / f'testdata_curriculum_stage_{stage:02d}_epoch_{epoch:05d}.pth'
        torch.save({
            'epoch': epoch, 'stage': stage, 'max_shift_pixels': stage,
            'model_state_dict': model3D.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'training_loss': loss.item(), 'image_loss': image_loss.item(), 'dvf_loss': dvf_loss.item(),
            'pretrained_model_path': str(PRETRAINED_MODEL_PATH), 'testdata_root': str(DATA_ROOT),
            'source_files': source_files,
        }, checkpoint_path)
        with open(OUTPUT_DIR / 'training_history.csv', 'w', newline='', encoding='utf-8') as handle:
            writer = csv.DictWriter(handle, fieldnames=history[0].keys())
            writer.writeheader(); writer.writerows(history)
        print(f'Saved stage checkpoint: {checkpoint_path}')

final_path = OUTPUT_DIR / 'model_analysis_pipeline_testdata_curriculum_final.pth'
torch.save({
    'epoch': TOTAL_EPOCHS, 'model_state_dict': model3D.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(), 'pretrained_model_path': str(PRETRAINED_MODEL_PATH),
    'testdata_root': str(DATA_ROOT), 'source_files': source_files,
}, final_path)
print(f'Completed {TOTAL_EPOCHS:,} epochs. Final model: {final_path.resolve()}')


In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import glob
from skimage.metrics import structural_similarity as ssim
from tabulate import tabulate
from piq import fsim

# ======================
# 定数（ここで評価したいスライス範囲を指定）
# ======================
CT_MIN, CT_MAX = -1200.0, 3146.98
SLICE_SIZE = (64, 128, 128)
SLICE_IDX_VIEW = 32    # デフォルト表示スライス（範囲指定がある場合は中央スライスが使われる）
SLICE_IDX_FSIM = 28

# 評価したいスライス範囲を指定（例: 20〜40）
# None にすると全ボリュームで評価します
SLICE_START = None
SLICE_END = None

# ======================
# 評価関数（変更なし）
# ======================
def compute_rmse(vol1, vol2):
    return np.sqrt(np.mean((vol1 - vol2) ** 2))

import torch
import torch.nn.functional as F
import numpy as np

def _gaussian_kernel_3d(window_size=7, sigma=1.5, device='cpu', dtype=torch.float32):
    coords = torch.arange(window_size, device=device, dtype=dtype) - window_size // 2
    g1 = torch.exp(-(coords**2) / (2 * sigma**2))
    g1 = g1 / g1.sum()
    g3 = g1[:, None, None] * g1[None, :, None] * g1[None, None, :]
    return g3.unsqueeze(0).unsqueeze(0)  # shape (1,1,D,H,W)

def _ssim_components_3d(x, y, window, K1=0.01, K2=0.03, eps=1e-12):
    """
    Compute luminance, contrast, structure maps for 3D volumes.
    x,y : tensors with shape (N,1,D,H,W)
    window : gaussian kernel shape (1,1,d,h,w)
    returns (l_map, cs_map) where cs_map = contrast * structure
    """
    pad = tuple([s//2 for s in window.shape[-3:]])
    mu_x = F.conv3d(x, window, padding=pad)
    mu_y = F.conv3d(y, window, padding=pad)

    mu_x_sq = mu_x * mu_x
    mu_y_sq = mu_y * mu_y
    mu_xy = mu_x * mu_y

    sigma_x_sq = F.conv3d(x * x, window, padding=pad) - mu_x_sq
    sigma_y_sq = F.conv3d(y * y, window, padding=pad) - mu_y_sq
    sigma_xy = F.conv3d(x * y, window, padding=pad) - mu_xy

    sigma_x_sq = torch.clamp(sigma_x_sq, min=0.0)
    sigma_y_sq = torch.clamp(sigma_y_sq, min=0.0)

    sigma_x = torch.sqrt(sigma_x_sq + eps)
    sigma_y = torch.sqrt(sigma_y_sq + eps)

    # Estimate L (data range) per-sample for numeric C's
    N = x.shape[0]
    max_x = x.view(N, -1).max(dim=1)[0].view(N,1,1,1,1)
    min_x = x.view(N, -1).min(dim=1)[0].view(N,1,1,1,1)
    max_y = y.view(N, -1).max(dim=1)[0].view(N,1,1,1,1)
    min_y = y.view(N, -1).min(dim=1)[0].view(N,1,1,1,1)
    L = torch.max(max_x - min_x, max_y - min_y)
    L = torch.clamp(L, min=eps)

    C1 = (K1 * L) ** 2
    C2 = (K2 * L) ** 2
    C3 = C2 / 2.0

    l_map = (2.0 * mu_xy + C1) / (mu_x_sq + mu_y_sq + C1 + eps)
    contrast = (2.0 * sigma_x * sigma_y + C2) / (sigma_x_sq + sigma_y_sq + C2 + eps)
    structure = (sigma_xy + C3) / (sigma_x * sigma_y + C3 + eps)

    cs_map = contrast * structure
    return l_map.clamp(min=eps), cs_map.clamp(min=eps)

def ms_ssim_3d(vol1, vol2,
               window_size=7, sigma=1.5,
               levels=5,
               weights=None,
               K1=0.01, K2=0.03, eps=1e-12,
               device=None):
    """
    Multi-scale SSIM for 3D volumes.
    vol1, vol2 : torch tensor or numpy array, shapes supported:
        (N,C,D,H,W), (C,D,H,W), (D,H,W)
    levels : number of scales (default 5)
    weights : list/tuple of length `levels` of weights summing to 1 (if None, use default MS-SSIM weights)
    returns: python float (mean MS-SSIM over batch)
    """
    # default MS-SSIM weights from the original paper (for 5 scales)
    if weights is None:
        # note: these are standard for 5-scale MS-SSIM
        weights = [0.0448, 0.2856, 0.3001, 0.2363, 0.1333]
    if len(weights) != levels:
        raise ValueError("len(weights) must equal levels")

    # convert numpy -> torch
    is_numpy = isinstance(vol1, np.ndarray) or isinstance(vol2, np.ndarray)
    if is_numpy:
        vol1 = torch.from_numpy(np.array(vol1))
        vol2 = torch.from_numpy(np.array(vol2))

    if not torch.is_tensor(vol1) or not torch.is_tensor(vol2):
        raise TypeError("vol1/vol2 must be numpy or torch tensor")

    # determine device
    if device is None:
        device = vol1.device if hasattr(vol1, 'device') else torch.device('cpu')
    device = torch.device(device)

    vol1 = vol1.to(device=device, dtype=torch.float32)
    vol2 = vol2.to(device=device, dtype=torch.float32)

    # normalize shapes to (N,C,D,H,W)
    def _ensure5d(x):
        if x.dim() == 5:
            return x
        if x.dim() == 4:
            return x.unsqueeze(0)
        if x.dim() == 3:
            return x.unsqueeze(0).unsqueeze(0)
        raise ValueError("Unsupported tensor shape: {}".format(x.shape))

    x = _ensure5d(vol1)
    y = _ensure5d(vol2)

    # unify channels by mean (if needed)
    if x.shape[1] != y.shape[1] or x.shape[1] > 1:
        x = x.mean(dim=1, keepdim=True)
        y = y.mean(dim=1, keepdim=True)

    N, C, D, H, W = x.shape

    # create gaussian kernel once
    window = _gaussian_kernel_3d(window_size=window_size, sigma=sigma, device=device, dtype=x.dtype)

    mcs = []   # mean contrast-structure per scale
    for lvl in range(levels):
        # if volume becomes too small to downsample, stop early
        if min(D, H, W) < 2:
            # compute final scale components and break
            l_map, cs_map = _ssim_components_3d(x, y, window, K1=K1, K2=K2, eps=eps)
            mcs.append(cs_map.view(N, -1).mean(dim=1))  # shape (N,)
            break

        l_map, cs_map = _ssim_components_3d(x, y, window, K1=K1, K2=K2, eps=eps)
        # mean over spatial dims gives per-sample scalar
        mcs.append(cs_map.view(N, -1).mean(dim=1))

        # downsample for next scale using average pooling (anti-aliasing)
        # kernel_size=2, stride=2
        x = F.avg_pool3d(x, kernel_size=2, stride=2, padding=0)
        y = F.avg_pool3d(y, kernel_size=2, stride=2, padding=0)
        N, C, D, H, W = x.shape

    # last scale luminance
    l_map, cs_map = _ssim_components_3d(x, y, window, K1=K1, K2=K2, eps=eps)
    l_mean = l_map.view(N, -1).mean(dim=1)  # per-sample

    # If we generated fewer mcs than requested levels (due to small volume), adjust weights
    actual_levels = len(mcs)
    if actual_levels < levels:
        # use last `actual_levels` weights and normalized
        w = torch.tensor(weights[:actual_levels], device=device, dtype=x.dtype)
        w = w / w.sum()
        used_weights = w
        # last weight for luminance is taken as original last weight mapped proportionally
        lum_weight = weights[min(actual_levels-1, len(weights)-1)]
    else:
        used_weights = torch.tensor(weights[:levels-1], device=device, dtype=x.dtype)  # weights for cs scales except last
        lum_weight = weights[levels-1]

    # compute MS-SSIM per sample:
    # product over scales of (mcs_i ^ weight_i)  and multiply by (l_mean ^ lum_weight)
    # convert list of tensors to shape (actual_levels, N)
    mcs_t = torch.stack(mcs[:actual_levels], dim=0)  # shape (actual_levels, N)
    # choose weights for these mcs: if actual_levels == levels -> first (levels-1) are cs weights, last is cs too
    if actual_levels == levels:
        cs_weights = torch.tensor(weights[:levels-1], device=device, dtype=x.dtype)
        # there are levels-1 cs entries and last is l_mean
        # mcs_t has length levels-1 (we appended cs for each scale before downsample), plus we still have final cs? 
        # Here we've appended cs at each level before downsampling, and computed final l_map after last downsample.
        # For typical MS-SSIM: use cs means from all levels and l only from last one.
        # So mcs_t currently includes cs for all levels (length == levels). In our loop we appended cs each iteration, and after final downsample we appended final cs too.
        # To match standard weights: use weights[:levels] for cs and lum_weight for l.
        cs_weights = torch.tensor(weights[:actual_levels], device=device, dtype=x.dtype)
    else:
        # when smaller, distribute weights proportionally — use used_weights for cs
        cs_weights = used_weights

    # Ensure cs_weights length matches mcs_t length
    if cs_weights.numel() != mcs_t.shape[0]:
        # simple fallback: evenly distribute
        cs_weights = torch.ones(mcs_t.shape[0], device=device, dtype=x.dtype) / float(mcs_t.shape[0])

    # raise mcs to weights and multiply (per-sample)
    # mcs_t ** cs_weights[:,None] -> shape (L, N)
    # product over rows -> (N,)
    ms_prod = torch.prod(mcs_t.pow(cs_weights.view(-1,1)), dim=0)
    ms_l = l_mean.pow(float(lum_weight))
    ms_per_sample = ms_prod * ms_l

    # return mean over batch as python float
    return float(ms_per_sample.mean().item())

def compute_fsim(vol1, vol2, device):
    """中央スライスでFSIMを評価（ただしこのスクリプトでは通常中央は範囲中央値に合わせる）"""
    v1 = vol1
    v2 = vol2
    # v1, v2 expected shape (C, D, H, W) or (D, H, W). Normalize access:
    if np.asarray(v1).ndim == 3:
        D = v1.shape[0]
    else:
        D = np.asarray(v1).shape[1]
    idx = min(SLICE_IDX_FSIM, max(0, D - 1))
    # Build tensors robustly for either shape
    if np.asarray(v1).ndim == 3:
        fixed_tensor = torch.tensor(v1[idx]).unsqueeze(0).unsqueeze(0).float().to(device)
        pred_tensor = torch.tensor(v2[idx]).unsqueeze(0).unsqueeze(0).float().to(device)
    else:
        fixed_tensor = torch.tensor(v1[:, idx, :, :]).unsqueeze(1).float().to(device)
        pred_tensor = torch.tensor(v2[:, idx, :, :]).unsqueeze(1).float().to(device)
        fixed_tensor = fixed_tensor[0].unsqueeze(0)
        pred_tensor = pred_tensor[0].unsqueeze(0)
    try:
        return fsim(pred_tensor, fixed_tensor, data_range=1.0, chromatic=False).item()
    except Exception:
        return float('nan')

def compute_ncc(vol1, vol2):
    v1 = vol1.flatten().astype(np.float32)
    v2 = vol2.flatten().astype(np.float32)
    v1_mean = v1.mean()
    v2_mean = v2.mean()
    numerator = np.sum((v1 - v1_mean) * (v2 - v2_mean))
    denominator = np.sqrt(np.sum((v1 - v1_mean) ** 2) * np.sum((v2 - v2_mean) ** 2) + 1e-8)
    return numerator / denominator

def compute_metrics(fixed, transformed, device, model_name):
    """Dice, Jaccard, SSIM, FSIM, RMSE, NCC をまとめて計算（入力は部分ボリュームでも可）"""
    fixed_bin = (fixed > 0.144).astype(int)
    transformed_bin = (transformed > 0.144).astype(int)

    dice = 2.0 * np.logical_and(fixed_bin, transformed_bin).sum() / (fixed_bin.sum() + transformed_bin.sum() + 1e-8)
    jaccard = np.logical_and(fixed_bin, transformed_bin).sum() / (np.logical_or(fixed_bin, transformed_bin).sum() + 1e-8)
#     ssim_val = compute_ssim_3d(fixed, transformed)
    ssim_val = ms_ssim_3d(fixed, transformed, levels=5, device='cpu')
    fsim_val = compute_fsim(fixed, transformed, device)
    ncc_val = compute_ncc(fixed, transformed)

    # RMSE用にCT値を復元
    pred_denorm = transformed * (CT_MAX - CT_MIN) + CT_MIN
    true_denorm = fixed * (CT_MAX - CT_MIN) + CT_MIN
    rmse_val = compute_rmse(true_denorm, pred_denorm)

    return [model_name, dice, jaccard, ssim_val, fsim_val, rmse_val, ncc_val]

# ================================================================
# Wavelet-model evaluation on the same TestData pairs as 128model_Test.ipynb
# ================================================================
# Run this cell after the training cell. It evaluates in exactly the same resized
# space as 128model_Test.ipynb: (64, 128, 128), slices 0..63 by default.
EVALUATION_MODEL_PATH = OUTPUT_DIR / 'model_analysis_pipeline_testdata_curriculum_final.pth'
EVAL_SLICE_START = 0
EVAL_SLICE_END = 63

if EVALUATION_MODEL_PATH.is_file():
    try:
        eval_checkpoint = torch.load(EVALUATION_MODEL_PATH, map_location=device, weights_only=True)
    except TypeError:
        eval_checkpoint = torch.load(EVALUATION_MODEL_PATH, map_location=device)
    eval_state = eval_checkpoint['model_state_dict'] if isinstance(eval_checkpoint, dict) and 'model_state_dict' in eval_checkpoint else eval_checkpoint
    model3D.load_state_dict(eval_state)
    print(f'Loaded evaluation weights: {EVALUATION_MODEL_PATH.resolve()}')
else:
    print('Final checkpoint not found; evaluating the model currently held in memory.')


def wavelet_register_fullres(moving_full, fixed_full):
    """Register a full-resolution pair and return reconstructed image + low-res flow."""
    moving_bands = down_sampling_3d(analysis_filter_3d(moving_full, analysis))
    fixed_bands = down_sampling_3d(analysis_filter_3d(fixed_full, analysis))
    flow = model3D(moving_bands, fixed_bands)
    warped_bands = torch.cat(
        [transformer(moving_bands[:, i:i + 1], flow) for i in range(moving_bands.shape[1])], dim=1
    )
    warped_full, _ = synthesis_filter_3d(up_sampling_3d(warped_bands), synthesis_filters)
    return warped_full, flow


def evaluate_wavelet_pair(pair_dir):
    npz_files = sorted(pair_dir.glob('*.npz'))
    if len(npz_files) != 2:
        raise ValueError(f'{pair_dir}: expected exactly 2 npz files, found {len(npz_files)}')

    # Keep the identical fixed/moving ordering used in 128model_Test.ipynb.
    with np.load(npz_files[0], allow_pickle=False) as archive:
        fixed_full_np = to_n_dhw(archive['Train'], str(npz_files[0]))[0]
    with np.load(npz_files[1], allow_pickle=False) as archive:
        moving_full_np = to_n_dhw(archive['Train'], str(npz_files[1]))[0]
    fixed_full = torch.from_numpy(fixed_full_np).unsqueeze(0).unsqueeze(0).to(device)
    moving_full = torch.from_numpy(moving_full_np).unsqueeze(0).unsqueeze(0).to(device)

    model3D.eval()
    with torch.no_grad():
        warped_full, flow = wavelet_register_fullres(moving_full, fixed_full)
        # This is the exact evaluation resolution of 128model_Test.ipynb.
        fixed_eval = F.interpolate(fixed_full, size=SLICE_SIZE, mode='trilinear', align_corners=False)
        moved_eval = F.interpolate(warped_full, size=SLICE_SIZE, mode='trilinear', align_corners=False)

    print(flow.shape)
    print(moved_eval.shape)
    fixed_np = fixed_eval[0, 0].cpu().numpy()
    moved_np = moved_eval[0, 0].cpu().numpy()
    s0 = max(0, EVAL_SLICE_START)
    s1 = min(fixed_np.shape[0] - 1, EVAL_SLICE_END)
    result = compute_metrics(
        fixed_np[s0:s1 + 1][np.newaxis, ...],
        moved_np[s0:s1 + 1][np.newaxis, ...],
        device,
        'Curriculum+TestData',
    )
    print(f'\n=== 📂 ペア: {pair_dir.name} の評価結果 (slices {s0}..{s1}) ===')
    print(tabulate(
        [result],
        headers=['モデル名', 'Dice', 'Jaccard', 'SSIM', 'FSIM', 'RMSE', 'NCC'],
        floatfmt='.4f', tablefmt='github',
    ))
    return result


all_results = []
for pair_dir in sorted(path for path in DATA_ROOT.iterdir() if path.is_dir()):
    all_results.append(evaluate_wavelet_pair(pair_dir))

if all_results:
    values = np.asarray([row[1:] for row in all_results], dtype=np.float64)
    print('\n=== 📊 全ペアの平均結果 ===')
    print(tabulate(
        [['Curriculum+TestData', *values.mean(axis=0)]],
        headers=['モデル名', '平均Dice', '平均Jaccard', '平均SSIM', '平均FSIM', '平均RMSE', '平均NCC'],
        floatfmt='.4f', tablefmt='github',
    ))
